# Risk-calibrated payload selection for spatial-domain steganography under detector mismatch — full study

A sender embeds with HILL costs in grayscale 256×256 BOSSbase covers. For each cover she chooses a payload with a rule
calibrated by **conformal risk control (CRC)** so that the detection risk measured by her own detectors stays below a target ε.
The warden, however, uses detectors the sender never sees. This notebook measures what the CRC guarantee buys against those
wardens and how much payload remains once the sender corrects for the mismatch with information she actually has.

| Item | Status | Question |
|---|---|---|
| **G** | Theorem (CRC), checked numerically | Mean test risk against the sender's detector family ≤ ε over random calibration/test splits. |
| **P1** | Proposition (exact identity) | With abstention, the risk of the images actually sent is R_sent = R_all / P(send). |
| **P2** | Proposition (exact bound) | Discarding the fraction q of smoothest covers is detectable from one image with AUC at most 0.5 + q/2. |
| **H3** | Hypothesis | Without correction, does the risk measured by independent wardens stay ≤ ε + 0.05? |
| **M** | Procedure + hypothesis | A mismatch margin set by leave-one-detector-out inside the sender's family: does it bring the warden risk to ≤ ε? |
| **H1** | Hypothesis | At the corrected operating points, which payload-allocation rule carries the most payload at a warden risk within ε: score-driven per-image selection, the detectability-limited sender, or one calibrated fixed payload? |

**Sender side.** A family of three detectors with different inductive biases (SRNet-lite, HPF-Net, Xu-style net); the calibration
statistic is the worst case over the family (maximum of standardized scores).
**Warden side.** Two detectors never queried by the sender: a strided-convolution network and an HPF-Net with its own
initialization and training draws.

**Baselines.** (a) one payload for every cover, calibrated by the same CRC criterion; (b) a detectability-limited sender in the
sense of Sedighi, Cogranne and Fridrich (IEEE TIFS 11(2):221–234, 2016): each cover receives the largest payload whose
model-based deflection stays below a bound, and the bound is calibrated by the same CRC criterion.

**Execution environment.** One GPU in a JupyterLab session (an HPC cluster or Colab).
Every trained network and every score lattice is saved, so an interrupted session resumes where it stopped.

In [ ]:
# ============================================================
# 1. Configuration — edit this cell only
# ============================================================
import os
from pathlib import Path

PROJECT_NAME = "RiskCalibratedStego_Full"

# BOSSbase 1.01: folder that contains the 10,000 .pgm files (or the zip one level up).
DATA_ROOT  = Path(os.environ.get("BOSS_DIR", Path.home() / "scratch" / "Datasets" / "BOSSbase_1.01")).expanduser()
BOSS_URL   = "https://dde.binghamton.edu/download/ImageDB/BOSSbase_1.01.zip"   # used only if DATA_ROOT is empty

# Large working files (image cache, score lattices, model weights): fast local/scratch disk.
WORK_DIR   = Path(os.environ.get("WORK_DIR", Path.home() / "scratch" / "Outputs" / PROJECT_NAME)).expanduser()
# Small result files (JSON, LaTeX tables, figures): copied here at the end.
RESULT_DIR = Path(os.environ.get("RESULT_DIR", Path.home() / "Outputs" / PROJECT_NAME)).expanduser()

CFG = dict(
    seed=2026,
    image_size=256,
    n_images=10000,
    n_surr_train=6000,         # detector training covers (sender family and wardens)
    n_surr_val=1000,           # validation covers: early stopping, admission rule, and the mismatch margin (Section 11)
    n_cal=1500,                # calibration covers
    n_test=1500,               # test covers
    payload_grid=[round(0.025 * k, 3) for k in range(0, 33)],   # 0.000 ... 0.800 bpp
    payload_floor=0.05,        # minimum payload of every transmitted image
    train_payload_range=(0.05, 0.80),
    family=["srnet_lite", "hpf_net", "xu_net"],      # sender-side detectors (calibration statistic = worst case)
    wardens=["warden_cnn", "warden_hpf_seed"],       # independent wardens, never queried by the sender
    warmup_payload=0.60,       # phase 1: fixed payload until the network leaves chance level
    warmup_target_acc=0.65,
    warmup_max_epochs=30,      # a network that never reaches the target is excluded
    batch_pairs=16,
    lr=1e-3,
    max_epochs=80,
    smooth=0.5,                # exponential smoothing of the validation loss used for stopping and LR decay
    min_delta=0.002,           # early stopping: minimum decrease of the smoothed validation loss ...
    patience=8,                # ... within this many epochs
    lr_patience=3, lr_factor=0.5,
    cover_fa=0.05,             # detector thresholds: 5% false alarms on clean covers
    eps_grid=[0.10, 0.20, 0.30],
    eps_int_step=0.01,         # grid of internal targets for the mismatch margin
    admission_q_grid=[0.00, 0.05, 0.10, 0.15, 0.20],
    admission_q_main=0.10,
    n_resplits=100,
    n_margin_resplits=50,
    eval_batch=32,
)

SMOKE_TEST = False
if SMOKE_TEST:
    CFG.update(n_images=160, n_surr_train=60, n_surr_val=20, n_cal=40, n_test=40,
               max_epochs=2, warmup_max_epochs=1, warmup_target_acc=0.0, n_resplits=10, n_margin_resplits=5,
               payload_grid=[0.0, 0.1, 0.2, 0.4, 0.6, 0.8], batch_pairs=2, eval_batch=4, patience=1)

assert CFG["n_surr_train"] + CFG["n_surr_val"] + CFG["n_cal"] + CFG["n_test"] <= CFG["n_images"]
for d in [WORK_DIR, WORK_DIR / "models", WORK_DIR / "cache"]:
    d.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT :", DATA_ROOT)
print("WORK_DIR  :", WORK_DIR)
print("RESULT_DIR:", RESULT_DIR)
print("Payload grid:", CFG["payload_grid"])

In [ ]:
# ============================================================
# Imports, device, reproducibility
# ============================================================
import json, math, time, zipfile, shutil, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU only")
print("torch", torch.__version__, "| numpy", np.__version__)
if DEVICE.type != "cuda" and not SMOKE_TEST:
    print("WARNING: no GPU visible. Request a GPU for this session; surrogate training on CPU is impractical.")

T0 = time.time()
def log(msg):
    print(f"[{(time.time() - T0) / 60:7.1f} min] {msg}", flush=True)

## 2. Covers: BOSSbase 1.01, resized to 256×256

If `DATA_ROOT` holds no `.pgm` files, the cell tries to download and unzip BOSSbase. On an Alliance cluster run
the download on a **login node** (compute nodes have no internet), e.g. in a terminal:
`mkdir -p ~/scratch/Datasets && cd ~/scratch/Datasets && wget <BOSS_URL> && unzip BOSSbase_1.01.zip`.
The resized images are cached once as a single `uint8` array.

In [ ]:
# ============================================================
# 2. Load (and if needed download) BOSSbase, resize to 256x256, cache
# ============================================================
def find_pgms(root):
    return sorted(Path(root).rglob("*.pgm")) if Path(root).exists() else []

pgms = find_pgms(DATA_ROOT)
if len(pgms) == 0:
    log(f"No .pgm files under {DATA_ROOT}; trying to download {BOSS_URL}")
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    zpath = DATA_ROOT.parent / "BOSSbase_1.01.zip"
    if not zpath.exists():
        def _hook(n, bs, total):
            if n % 2000 == 0 and total > 0:
                print(f"   downloaded {min(n * bs, total) / 1e6:8.1f} / {total / 1e6:.1f} MB", flush=True)
        urllib.request.urlretrieve(BOSS_URL, zpath, _hook)
    log("Unzipping ...")
    with zipfile.ZipFile(zpath) as z:
        z.extractall(DATA_ROOT.parent)
    pgms = find_pgms(DATA_ROOT.parent)
assert len(pgms) >= CFG["n_images"], f"found {len(pgms)} .pgm files, need {CFG['n_images']}"
pgms = pgms[: CFG["n_images"]]
log(f"Found {len(pgms)} cover files")

cache = WORK_DIR / "cache" / f"boss_{CFG['n_images']}_{CFG['image_size']}.npy"
if cache.exists():
    IMGS = np.load(cache)
    log(f"Loaded cached array {IMGS.shape}")
else:
    S = CFG["image_size"]
    IMGS = np.empty((len(pgms), S, S), dtype=np.uint8)
    for i, p in enumerate(pgms):
        im = Image.open(p).convert("L")
        if im.size != (S, S):
            im = im.resize((S, S), Image.BICUBIC)
        IMGS[i] = np.asarray(im, dtype=np.uint8)
        if (i + 1) % 1000 == 0 or i + 1 == len(pgms):
            log(f"   resized {i + 1}/{len(pgms)}")
    np.save(cache, IMGS)
    log(f"Cached {IMGS.shape} to {cache}")

In [ ]:
# ============================================================
# 3. Disjoint splits: surrogate train / surrogate val / calibration / test
# ============================================================
rng = np.random.default_rng(CFG["seed"])
perm = rng.permutation(len(IMGS))
a = CFG["n_surr_train"]; b = a + CFG["n_surr_val"]; c = b + CFG["n_cal"]; d = c + CFG["n_test"]
IDX = dict(train=perm[:a], val=perm[a:b], cal=perm[b:c], test=perm[c:d])
for k, v in IDX.items():
    print(f"{k:5s}: {len(v)} images")
assert len(set(np.concatenate(list(IDX.values())))) == d, "splits overlap"

## 4. HILL costs and a nested ternary embedding simulator

* **HILL costs:** ρ = (1 / (|X ⊛ K_KB| ⊛ L₃)) ⊛ L₁₅ with the 3×3 Ker–Böhme high-pass kernel and 3×3 / 15×15 averaging filters.
* **Simulator:** optimal ternary change probabilities p± = e^{−λρ}/(1 + 2e^{−λρ}); λ is found per image by bisection so that the
  ternary entropy equals the payload α·H·W bits.
* **Saturated pixels** are handled with boolean masks (a pixel at 0 cannot go to −1, a pixel at 255 cannot go to +1), not with
  large costs, so no overflow can occur.
* **Nested embeddings:** each image has one fixed uniform field U. A pixel changes by +1 if U < p₊ and by −1 if U > 1 − p₋.
  Because p± grow with the payload, the set of changed pixels grows monotonically with α for a given image.

In [ ]:
# ============================================================
# 4. HILL costs + nested ternary embedding simulator (GPU, batched)
# ============================================================
KB = torch.tensor([[-1., 2., -1.], [2., -4., 2.], [-1., 2., -1.]]) / 4.0

def hill_costs(x):
    # x: (B,1,H,W) float in [0,255] -> costs (B,1,H,W)
    k = KB.to(x.device).view(1, 1, 3, 3)
    r = torch.abs(F.conv2d(F.pad(x, (1, 1, 1, 1), mode="reflect"), k))
    r = F.avg_pool2d(F.pad(r, (1, 1, 1, 1), mode="reflect"), 3, stride=1)
    rho = 1.0 / (r + 1e-10)
    rho = F.avg_pool2d(F.pad(rho, (7, 7, 7, 7), mode="reflect"), 15, stride=1)
    return rho.clamp(max=1e8)

def _probs(rho, lam, m_plus, m_minus):
    w = torch.exp(-lam * rho)                    # lam*rho >= 0 -> no overflow
    wp, wm = w * m_plus, w * m_minus
    z = 1.0 + wp + wm
    return wp / z, wm / z

def _entropy_bits(pp, pm):
    p0 = (1.0 - pp - pm).clamp(min=0.0)
    h = -(torch.special.xlogy(pp, pp) + torch.special.xlogy(pm, pm) + torch.special.xlogy(p0, p0))
    return h.sum(dim=(1, 2, 3)) / math.log(2.0)

@torch.no_grad()
def change_probs(x, alpha, n_iter=60):
    # x: (B,1,H,W) float; alpha: (B,) payload in bpp -> p_plus, p_minus (B,1,H,W)
    B, _, H, W = x.shape
    rho = hill_costs(x)
    m_plus = (x < 255).float(); m_minus = (x > 0).float()
    target = alpha.to(x.device).float() * H * W
    lo = torch.full((B,), -12.0, device=x.device)    # log10(lambda)
    hi = torch.full((B,), 6.0, device=x.device)
    for _ in range(n_iter):
        mid = 0.5 * (lo + hi)
        lam = (10.0 ** mid).view(B, 1, 1, 1)
        pp, pm = _probs(rho, lam, m_plus, m_minus)
        h = _entropy_bits(pp, pm)
        too_much = h > target                         # entropy decreases with lambda
        lo = torch.where(too_much, mid, lo)
        hi = torch.where(too_much, hi, mid)
    lam = (10.0 ** hi).view(B, 1, 1, 1)
    pp, pm = _probs(rho, lam, m_plus, m_minus)
    zero = (alpha <= 0).to(x.device).view(B, 1, 1, 1)
    return pp.masked_fill(zero, 0.0), pm.masked_fill(zero, 0.0)

@torch.no_grad()
def embed(x, alpha, u):
    # Nested simulated embedding: x (B,1,H,W) float, alpha (B,), u (B,1,H,W) uniform field
    pp, pm = change_probs(x, alpha)
    delta = (u < pp).float() - (u > 1.0 - pm).float()
    return (x + delta).clamp(0, 255)

def uniform_field(img_ids, shape):
    # Deterministic per-image uniform field -> identical U for every payload of the same image
    out = np.empty((len(img_ids), 1) + shape, dtype=np.float32)
    for j, i in enumerate(img_ids):
        out[j, 0] = np.random.default_rng(CFG["seed"] * 100003 + int(i)).random(shape, dtype=np.float32)
    return torch.from_numpy(out)

# --- self-test: payload accuracy and nesting
_x = torch.from_numpy(IMGS[IDX["val"][:4]].astype(np.float32)).unsqueeze(1).to(DEVICE)
_u = uniform_field(IDX["val"][:4], _x.shape[-2:]).to(DEVICE)
prev = None
for a_ in [0.1, 0.3, 0.5]:
    al = torch.full((4,), a_)
    pp, pm = change_probs(_x, al)
    bits = _entropy_bits(pp, pm) / (_x.shape[-1] * _x.shape[-2])
    y = embed(_x, al, _u)
    changed = (y != _x)
    nested = True if prev is None else bool((prev & ~changed).sum() == 0)
    print(f"alpha={a_:.2f}  achieved bpp={bits.cpu().numpy().round(4)}  change rate={changed.float().mean().item():.4f}  nested={nested}")
    prev = changed

## 5. Detectors

All networks use **GroupNorm** instead of BatchNorm, so a network behaves identically in training and evaluation mode
(with BatchNorm, running statistics can make the validation loss fluctuate strongly between epochs).

* **Sender family:** SRNet-lite (unpooled residual layers, learned front end), HPF-Net (fixed high-pass bank + truncation),
  Xu-style net (KV high-pass filter, absolute-value layer, tanh, 5×5 then 1×1 convolutions).
* **Wardens:** `warden_cnn` (high-pass bank, strided convolutions, ELU, no truncation) and `warden_hpf_seed`
  (HPF-Net architecture, own initialization and training draws). Wardens train on the same public cover source, as a warden
  following Kerckhoffs' principle would.

Training: (1) warm-up at a fixed `warmup_payload` until validation accuracy reaches `warmup_target_acc` (networks that never do
are excluded); (2) payloads drawn uniformly in `train_payload_range`, learning rate halved when the smoothed validation loss stalls
for `lr_patience` epochs, and training stopped when it has not decreased by `min_delta` for `patience` epochs.

In [ ]:
# ============================================================
# 5. Detector architectures (GroupNorm everywhere)
# ============================================================
def NORM(c): return nn.GroupNorm(min(8, max(1, c // 2)), c)

class T1(nn.Module):
    def __init__(s, cin, cout):
        super().__init__(); s.c = nn.Conv2d(cin, cout, 3, padding=1, bias=False); s.b = NORM(cout)
    def forward(s, x): return F.relu(s.b(s.c(x)))

class T2(nn.Module):  # unpooled residual block
    def __init__(s, ch):
        super().__init__(); s.a = T1(ch, ch); s.c = nn.Conv2d(ch, ch, 3, padding=1, bias=False); s.b = NORM(ch)
    def forward(s, x): return x + s.b(s.c(s.a(x)))

class T3(nn.Module):  # pooled residual block
    def __init__(s, cin, cout):
        super().__init__()
        s.main = nn.Sequential(T1(cin, cout), nn.Conv2d(cout, cout, 3, padding=1, bias=False), NORM(cout),
                               nn.AvgPool2d(3, 2, 1))
        s.skip = nn.Sequential(nn.Conv2d(cin, cout, 1, stride=2, bias=False), NORM(cout))
    def forward(s, x): return s.main(x) + s.skip(x)

class SRNetLite(nn.Module):
    def __init__(s):
        super().__init__()
        s.f = nn.Sequential(T1(1, 64), T1(64, 16), *[T2(16) for _ in range(5)],
                            T3(16, 16), T3(16, 64), T3(64, 128), T3(128, 256))
        s.fc = nn.Linear(256, 1)
    def forward(s, x):
        x = (x - 128.0) / 64.0
        return s.fc(s.f(x).mean(dim=(2, 3))).squeeze(1)

def hpf_bank():
    k = []
    k.append([[-1, 2, -2, 2, -1], [2, -6, 8, -6, 2], [-2, 8, -12, 8, -2], [2, -6, 8, -6, 2], [-1, 2, -2, 2, -1]])  # KV
    kb = np.zeros((5, 5)); kb[1:4, 1:4] = [[-1, 2, -1], [2, -4, 2], [-1, 2, -1]]; k.append(kb)
    for (dy, dx) in [(0, 1), (1, 0), (1, 1), (1, -1)]:          # second-order residuals
        f = np.zeros((5, 5)); f[2, 2] = -2; f[2 + dy, 2 + dx] = 1; f[2 - dy, 2 - dx] = 1; k.append(f)
    for (dy, dx) in [(0, 1), (1, 0), (1, 1), (1, -1)]:          # first-order residuals
        f = np.zeros((5, 5)); f[2, 2] = -1; f[2 + dy, 2 + dx] = 1; k.append(f)
    k = np.array(k, dtype=np.float32)
    k /= np.abs(k).sum(axis=(1, 2), keepdims=True)
    return torch.from_numpy(k).unsqueeze(1)

class HPFNet(nn.Module):
    def __init__(s, tlu=3.0):
        super().__init__()
        s.register_buffer("hpf", hpf_bank()); s.tlu = tlu
        nk = s.hpf.shape[0]
        def blk(ci, co, pool):
            L = [nn.Conv2d(ci, co, 3, padding=1, bias=False), NORM(co), nn.ReLU()]
            if pool: L.append(nn.AvgPool2d(3, 2, 1))
            return L
        s.f = nn.Sequential(*blk(nk, 32, False), *blk(32, 32, False), *blk(32, 64, True),
                            *blk(64, 128, True), *blk(128, 256, True))
        s.fc = nn.Linear(256, 1)
    def forward(s, x):
        r = F.conv2d(F.pad(x, (2, 2, 2, 2), mode="reflect"), s.hpf).clamp(-s.tlu, s.tlu)
        return s.fc(s.f(r).mean(dim=(2, 3))).squeeze(1)

class XuNet(nn.Module):
    # KV high-pass filter, ABS layer, tanh in the first two groups, 1x1 convolutions afterwards
    def __init__(s):
        super().__init__()
        s.register_buffer("kv", hpf_bank()[:1])
        s.g1 = nn.Sequential(nn.Conv2d(1, 16, 5, padding=2, bias=False), NORM(16))
        s.g2 = nn.Sequential(nn.Conv2d(16, 32, 5, padding=2, bias=False), NORM(32), nn.Tanh(), nn.AvgPool2d(5, 2, 2))
        def g(ci, co): return nn.Sequential(nn.Conv2d(ci, co, 1, bias=False), NORM(co), nn.ReLU(), nn.AvgPool2d(5, 2, 2))
        s.rest = nn.Sequential(g(32, 64), g(64, 128), g(128, 256))
        s.fc = nn.Linear(256, 1)
    def forward(s, x):
        r = F.conv2d(F.pad(x, (2, 2, 2, 2), mode="reflect"), s.kv)
        h = torch.tanh(s.g1(r).abs()); h = F.avg_pool2d(h, 5, 2, 2)
        return s.fc(s.rest(s.g2(h)).mean(dim=(2, 3))).squeeze(1)

class WardenCNN(nn.Module):
    # No truncation, strided convolutions instead of average pooling, ELU activations
    def __init__(s):
        super().__init__()
        s.register_buffer("hpf", hpf_bank()); nk = s.hpf.shape[0]
        def blk(ci, co, stride):
            return [nn.Conv2d(ci, co, 3, stride=stride, padding=1, bias=False), NORM(co), nn.ELU()]
        s.f = nn.Sequential(*blk(nk, 48, 1), *blk(48, 48, 1), *blk(48, 96, 2), *blk(96, 96, 1),
                            *blk(96, 192, 2), *blk(192, 192, 2), *blk(192, 256, 2))
        s.fc = nn.Linear(256, 1)
    def forward(s, x):
        r = F.conv2d(F.pad(x, (2, 2, 2, 2), mode="reflect"), s.hpf)
        return s.fc(s.f(r).mean(dim=(2, 3))).squeeze(1)

NETS = {  # name: (constructor, init seed, payload-draw seed)
    "srnet_lite":      (SRNetLite, CFG["seed"] + 303, CFG["seed"] + 3000),
    "hpf_net":         (HPFNet,    CFG["seed"],       CFG["seed"]),
    "xu_net":          (XuNet,     CFG["seed"] + 404, CFG["seed"] + 4000),
    "warden_cnn":      (WardenCNN, CFG["seed"] + 101, CFG["seed"] + 5000),
    "warden_hpf_seed": (HPFNet,    CFG["seed"] + 202, CFG["seed"] + 9000),
}
for n in CFG["family"] + CFG["wardens"]:
    print(f"{n:16s}: {sum(p.numel() for p in NETS[n][0]().parameters()) / 1e6:.2f} M parameters")

In [ ]:
# ============================================================
# 6. Train the sender family and the wardens
# ============================================================
def d4(x, k):
    if k >= 4: x = torch.flip(x, dims=[-1])
    return torch.rot90(x, k % 4, dims=[-2, -1])

def make_batch(ids, train, prange, gen):
    x = torch.from_numpy(IMGS[ids].astype(np.float32)).unsqueeze(1).to(DEVICE)
    lo, hi = prange
    alpha = torch.from_numpy(gen.uniform(lo, hi, size=len(ids)).astype(np.float32))
    u = torch.from_numpy(gen.random((len(ids), 1) + x.shape[-2:], dtype=np.float32)).to(DEVICE)
    y = embed(x, alpha, u)
    if train:
        k = int(gen.integers(8)); x, y = d4(x, k), d4(y, k)
    inp = torch.cat([x, y]); lab = torch.cat([torch.zeros(len(ids)), torch.ones(len(ids))]).to(DEVICE)
    return inp, lab

def run_epoch(model, ids, opt, prange, train, seed):
    gen = np.random.default_rng(seed)
    model.train(train)
    ids = gen.permutation(ids) if train else ids
    bp = CFG["batch_pairs"]; tot, n, correct = 0.0, 0, 0
    for s in range(0, len(ids), bp):
        inp, lab = make_batch(ids[s:s + bp], train, prange, gen)
        with torch.set_grad_enabled(train):
            out = model(inp); loss = F.binary_cross_entropy_with_logits(out, lab)
            if train:
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        tot += loss.item() * len(lab); n += len(lab); correct += ((out > 0).float() == lab).sum().item()
    return tot / n, correct / n

def train_net(name):
    ctor, init_seed, data_seed = NETS[name]
    path = WORK_DIR / "models" / f"{name}.pt"; status_path = WORK_DIR / "models" / f"{name}_status.json"
    torch.manual_seed(init_seed); model = ctor().to(DEVICE)
    if path.exists() and status_path.exists():
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        ok = json.loads(status_path.read_text())["warmup_ok"]
        log(f"{name}: loaded {path} (usable: {ok})"); return model.eval(), ok
    opt = torch.optim.Adamax(model.parameters(), lr=CFG["lr"])
    hist = []; wp = CFG["warmup_payload"]; warm_ok = False
    for ep in range(CFG["warmup_max_epochs"]):
        tl, ta = run_epoch(model, IDX["train"], opt, (wp, wp), True, data_seed + ep)
        vl, va = run_epoch(model, IDX["val"], None, (wp, wp), False, CFG["seed"] + 7)
        hist.append(dict(phase=1, epoch=ep, train_loss=tl, train_acc=ta, val_loss=vl, val_acc=va, lr=CFG["lr"]))
        log(f"{name} warm-up ep {ep:3d} @ {wp:.2f} bpp | train {tl:.4f}/{ta:.3f} | val {vl:.4f}/{va:.3f} | target acc {CFG['warmup_target_acc']:.2f}")
        if va >= CFG["warmup_target_acc"]:
            warm_ok = True; log(f"{name}: left chance level after {ep + 1} warm-up epochs"); break
    if not warm_ok:
        log(f"{name}: WARNING — validation accuracy never reached {CFG['warmup_target_acc']:.2f}; network not used")
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=CFG["lr_factor"], patience=CFG["lr_patience"],
                                                       threshold=CFG["min_delta"], threshold_mode="abs")
    best, wait, sm = float("inf"), 0, None
    torch.save(model.state_dict(), path)
    for ep in range(CFG["max_epochs"] if warm_ok else 0):
        tl, ta = run_epoch(model, IDX["train"], opt, CFG["train_payload_range"], True, data_seed + 1000 + ep)
        vl, va = run_epoch(model, IDX["val"], None, CFG["train_payload_range"], False, CFG["seed"] + 7)
        sm = vl if sm is None else CFG["smooth"] * sm + (1 - CFG["smooth"]) * vl
        sched.step(sm); lr = opt.param_groups[0]["lr"]
        hist.append(dict(phase=2, epoch=ep, train_loss=tl, train_acc=ta, val_loss=vl, val_acc=va, val_loss_smoothed=sm, lr=lr))
        if sm < best - CFG["min_delta"]:
            best, wait = sm, 0; torch.save(model.state_dict(), path)
        else:
            wait += 1
        log(f"{name} ep {ep:3d} | train {tl:.4f}/{ta:.3f} | val {vl:.4f}/{va:.3f} | smoothed {sm:.4f} | best {best:.4f} | lr {lr:.1e} | wait {wait}/{CFG['patience']}")
        if wait >= CFG["patience"]:
            log(f"{name}: stopped (smoothed val loss did not decrease by {CFG['min_delta']} for {CFG['patience']} epochs)"); break
    pd.DataFrame(hist).to_csv(WORK_DIR / f"train_history_{name}.csv", index=False)
    status_path.write_text(json.dumps(dict(warmup_ok=warm_ok)))
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    return model.eval(), warm_ok

# Cheaper networks first; every finished network is saved, so an interrupted session resumes with the rest
ORDER = ["hpf_net", "xu_net", "warden_hpf_seed", "warden_cnn", "srnet_lite"]
ORDER = [n for n in ORDER if n in CFG["family"] + CFG["wardens"]]
TRAINED = {n: train_net(n) for n in ORDER}
MODELS = {n: TRAINED[n][0] for n in CFG["family"] if TRAINED[n][1]}
WARDENS = {w: TRAINED[w][0] for w in CFG["wardens"] if TRAINED[w][1]}
assert len(MODELS) >= 1 and len(WARDENS) >= 1, "too few usable networks"
print("Sender family:", list(MODELS), "| wardens:", list(WARDENS))
if len(MODELS) < 2:
    print("WARNING: fewer than two family members — the leave-one-detector-out margin (Section 11) cannot be computed.")

## 7. Score lattices and model-based deflection

Every validation, calibration and test cover is embedded at every grid payload (nested embeddings) and scored by every
network. For the detectability-limited baseline the same pass records the model-based deflection
δ²(X, α) = Σᵢ 4 βᵢ² / σᵢ⁴, with βᵢ = (p₊ + p₋)/2 the change rate of pixel i and σᵢ² a local residual-variance estimate
(3×3 local-mean residual, 7×7 variance window, floor 0.01). This follows the multivariate Gaussian cover model of the
detectability-limited sender with a simplified variance estimator; only the ordering of covers by δ² matters, because its
bound is calibrated by CRC.

In [ ]:
# ============================================================
# 7. Score lattices (all networks) + deflection lattice, cached
# ============================================================
GRID = np.array(CFG["payload_grid"], dtype=np.float32)
EVAL_IDS = np.concatenate([IDX["val"], IDX["cal"], IDX["test"]])
nv, nc, nt = len(IDX["val"]), len(IDX["cal"]), len(IDX["test"])
POS = dict(val=np.arange(nv), cal=nv + np.arange(nc), test=nv + nc + np.arange(nt))

@torch.no_grad()
def local_var(x):
    r = x - F.avg_pool2d(F.pad(x, (1, 1, 1, 1), mode="reflect"), 3, stride=1)
    v = F.avg_pool2d(F.pad(r * r, (3, 3, 3, 3), mode="reflect"), 7, stride=1)
    return v.clamp(min=0.01)

@torch.no_grad()
def score_lattice(nets, ids, with_deflection):
    out = {n: np.zeros((len(ids), len(GRID)), dtype=np.float32) for n in nets}
    D = np.zeros((len(ids), len(GRID)), dtype=np.float64) if with_deflection else None
    eb = CFG["eval_batch"]
    for s in range(0, len(ids), eb):
        b = ids[s:s + eb]
        x = torch.from_numpy(IMGS[b].astype(np.float32)).unsqueeze(1).to(DEVICE)
        u = uniform_field(b, x.shape[-2:]).to(DEVICE)
        inv_s4 = 1.0 / local_var(x) ** 2 if with_deflection else None
        for k, a_ in enumerate(GRID):
            if a_ == 0:
                y = x
            else:
                al = torch.full((len(b),), float(a_))
                if with_deflection:
                    pp, pm = change_probs(x, al)
                    beta = 0.5 * (pp + pm)
                    D[s:s + len(b), k] = (4.0 * beta * beta * inv_s4).sum(dim=(1, 2, 3)).double().cpu().numpy()
                y = embed(x, al, u)
            for n, m in nets.items():
                out[n][s:s + len(b), k] = m(y).float().cpu().numpy()
        if (s // eb) % 10 == 0 or s + eb >= len(ids):
            log(f"   lattice {min(s + eb, len(ids))}/{len(ids)} images ({', '.join(nets) or 'deflection only'})")
    return out, D

def lattice_path(n): return WORK_DIR / "cache" / f"lattice_{n}_{len(EVAL_IDS)}_{len(GRID)}.npz"
ALL = {**MODELS, **WARDENS}; LATS = {}
for n in ALL:
    if lattice_path(n).exists():
        LATS[n] = np.load(lattice_path(n))["s"]; log(f"{n}: loaded lattice")
todo = {n: m for n, m in ALL.items() if n not in LATS}
need_D = not lattice_path("deflection").exists()
if todo or need_D:
    new, Dnew = score_lattice(todo, EVAL_IDS, need_D)
    for n, v in new.items():
        LATS[n] = v; np.savez(lattice_path(n), s=v)
    if need_D:
        np.savez(lattice_path("deflection"), s=Dnew)
DEFL = np.maximum.accumulate(np.load(lattice_path("deflection"))["s"], axis=1)   # non-decreasing in payload
LAT = {n: LATS[n] for n in MODELS}

def auc(neg, pos):
    # Mann-Whitney AUC via average ranks (ties handled)
    r = pd.Series(np.concatenate([pos, neg])).rank(method="average").to_numpy()
    return float((r[: len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))
rows = [dict(model=n, role="family" if n in MODELS else "warden", payload=float(a_),
             auc=auc(LATS[n][POS["test"], 0], LATS[n][POS["test"], k]))
        for n in ALL for k, a_ in enumerate(GRID) if k > 0]
NET_AUC = pd.DataFrame(rows)
print("AUC, clean vs stego test covers:")
print(NET_AUC.pivot(index="payload", columns="model", values="auc").round(3).to_string())

## 8. Calibration statistic and policies

* **Family statistic.** Each family logit is standardized with the median and IQR of its scores on clean calibration covers;
  the statistic is the maximum over the family, so an image counts as detected when any family member flags it. One joint
  threshold τ gives 5% false alarms on clean calibration covers.
* **Envelope.** z(X, α) = s(stego) − τ and z̃(X, α) = running maximum over a ≤ α (clean cover included), non-decreasing in α.
* **Policies** — each is "largest grid payload with SEL(X, α) ≤ t", with t calibrated by CRC on the loss 1{z̃(X, α) ≥ 0}:
  * **per-image:** SEL = z̃ (the family's own margin);
  * **fixed:** SEL = α (the same payload for every cover);
  * **detectability-limited:** SEL = δ²(X, α).

  For each policy the loss is non-decreasing in t, so CRC applies: t̂ = largest t with (n/(n+1))·R̂ₙ(t) + 1/(n+1) ≤ ε.
* **Transmission-conditioned** versions (the main setting) discard covers below a residual-energy admission threshold fixed on
  the validation covers, and send every admitted cover with at least `payload_floor`. The **abstaining** version (payload 0 allowed)
  is kept for G and P1.

In [ ]:
# ============================================================
# 8. Family statistic, envelope, generic CRC-calibrated policies
# ============================================================
def family_envelope(names, ref_pos):
    std = {}
    for n in names:
        c0 = LATS[n][ref_pos, 0]
        std[n] = (float(np.median(c0)), float(max(np.subtract(*np.percentile(c0, [75, 25])), 1e-6)))
    s = np.max(np.stack([(LATS[n] - std[n][0]) / std[n][1] for n in names]), axis=0)
    tau = float(np.quantile(s[ref_pos, 0], 1 - CFG["cover_fa"], method="higher"))
    return np.maximum.accumulate(s - tau, axis=1), std, tau

cal_c = POS["cal"]
ZT, STD, TAU = family_envelope(list(MODELS), cal_c)
print(f"family tau = {TAU:.3f} | clean-cover false alarms: cal {np.mean(ZT[cal_c, 0] >= 0):.3f}, test {np.mean(ZT[POS['test'], 0] >= 0):.3f}")
LOSS = (ZT >= 0)                                              # family flag at each payload (monotone in payload)
FLOOR_K = int(np.searchsorted(GRID, CFG["payload_floor"] - 1e-9))
SEL = dict(per_image=ZT, fixed=np.broadcast_to(GRID, ZT.shape), detectability_limited=DEFL)

def select(sel_rows, t, floor=True):
    ok = sel_rows <= t
    if floor:
        ok[:, :FLOOR_K] = False
        return np.where(ok.any(axis=1), ok.shape[1] - 1 - np.argmax(ok[:, ::-1], axis=1), FLOOR_K)
    ok[:, 0] = True                                           # abstaining: payload 0 always allowed
    return ok.shape[1] - 1 - np.argmax(ok[:, ::-1], axis=1)

def crc_t(sel_rows, loss_rows, eps, floor=True):
    # Largest t on the candidate grid satisfying the CRC criterion; None if even the minimum payload fails
    n = len(sel_rows); idx = np.arange(n)
    cand = np.unique(np.concatenate([[-np.inf], np.asarray(sel_rows).ravel()]))
    def crit(t):
        k = select(np.array(sel_rows), t, floor)
        lo = loss_rows[idx, k] & ((k > 0) if not floor else True)
        return n / (n + 1) * lo.mean() + 1 / (n + 1)
    if crit(-np.inf) > eps:
        return None
    lo, hi = 0, len(cand) - 1                                  # criterion non-decreasing in t -> bisection
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if crit(cand[mid]) <= eps: lo = mid
        else: hi = mid - 1
    return cand[lo]

# Admission rule: residual energy E(X), threshold = q-quantile on the validation covers (fixed beforehand)
KV = hpf_bank()[0, 0].clone()
@torch.no_grad()
def residual_energy(ids):
    out = []
    for s in range(0, len(ids), 256):
        x = torch.from_numpy(IMGS[ids[s:s + 256]].astype(np.float32)).unsqueeze(1).to(DEVICE)
        r = F.conv2d(F.pad(x, (2, 2, 2, 2), mode="reflect"), KV.view(1, 1, 5, 5).to(DEVICE))
        out.append(r.abs().mean(dim=(1, 2, 3)).cpu().numpy())
    return np.concatenate(out)
E_EVAL = residual_energy(EVAL_IDS); E_VAL = E_EVAL[POS["val"]]
def admit_threshold(q): return -np.inf if q <= 0 else float(np.quantile(E_VAL, q))
thr_main = admit_threshold(CFG["admission_q_main"])
print("Admission thresholds:", {q: round(admit_threshold(q), 3) for q in CFG["admission_q_grid"]})

# Warden flags: each warden's own threshold at 5% false alarms on clean calibration covers
W_TAU = {w: float(np.quantile(LATS[w][cal_c, 0], 1 - CFG["cover_fa"], method="higher")) for w in WARDENS}
W_FLAG = {w: LATS[w] >= W_TAU[w] for w in WARDENS}
for w in WARDENS:
    print(f"{w}: threshold {W_TAU[w]:.3f} | false alarms on clean test covers {W_FLAG[w][POS['test'], 0].mean():.3f}")

## 9. G and P1 — the guarantee against the family, and the cost of abstention

Calibration and test covers are pooled and re-split at random `n_resplits` times. The guaranteed quantity is the mean test
risk over the calibrated population; it is flagged if above ε + 2 SE. For the abstaining policy, P1 checks
R_sent = R_all / P(sent) numerically.

In [ ]:
# ============================================================
# 9. G (all policies) and P1 (abstaining policy)
# ============================================================
POOL = np.concatenate([POS["cal"], POS["test"]]); HALF = len(POOL) // 2
def resplit(r, base):
    p = np.random.default_rng(base + r).permutation(POOL); return p[:HALF], p[HALF:]

recs = []
for r in range(CFG["n_resplits"]):
    ca, te = resplit(r, 10_000)
    ca_a, te_a = ca[E_EVAL[ca] >= thr_main], te[E_EVAL[te] >= thr_main]
    for eps in CFG["eps_grid"]:
        t = crc_t(ZT[ca], LOSS[ca], eps, floor=False)
        k = select(ZT[te].copy(), t, floor=False); sent = k > 0
        loss = LOSS[te, k] & sent
        recs.append(dict(resplit=r, eps=eps, policy="abstaining", feasible=True, risk_all=loss.mean(),
                         p_sent=sent.mean(), risk_sent=loss[sent].mean() if sent.any() else np.nan,
                         mean_payload_sent=GRID[k][sent].mean() if sent.any() else 0.0))
        for pol, S in SEL.items():
            t = crc_t(S[ca_a], LOSS[ca_a], eps)
            if t is None:
                recs.append(dict(resplit=r, eps=eps, policy=pol, feasible=False)); continue
            k = select(np.array(S[te_a]), t); loss = LOSS[te_a, k]
            recs.append(dict(resplit=r, eps=eps, policy=pol, feasible=True, risk_all=loss.mean(),
                             p_sent=len(te_a) / len(te), risk_sent=loss.mean(), mean_payload_sent=GRID[k].mean()))
    if (r + 1) % 10 == 0:
        log(f"   G resplit {r + 1}/{CFG['n_resplits']}")
G_RAW = pd.DataFrame(recs)

rows = []
for (pol, eps), d in G_RAW.groupby(["policy", "eps"]):
    d = d[d.feasible == True]
    if len(d) == 0:
        rows.append(dict(policy=pol, eps=eps, feasible_frac=0.0)); continue
    m, se = d.risk_all.mean(), d.risk_all.std(ddof=1) / np.sqrt(len(d))
    rows.append(dict(policy=pol, eps=eps, feasible_frac=len(d) / CFG["n_resplits"], mean_risk_calibrated=m, se=se,
                     guarantee_ok=bool(m <= eps + 2 * se), mean_risk_sent=d.risk_sent.mean(), p_sent=d.p_sent.mean(),
                     mean_payload_sent_bpp=d.mean_payload_sent.mean()))
G_SUM = pd.DataFrame(rows)
pd.set_option("display.width", 220)
print(G_SUM.round(4).to_string(index=False))

ab = G_RAW[G_RAW.policy == "abstaining"]
P1 = ab.groupby("eps").apply(lambda d: pd.Series(dict(
        R_all=d.risk_all.mean(), P_sent=d.p_sent.mean(), R_sent=d.risk_sent.mean(),
        R_all_over_P_sent=(d.risk_all / d.p_sent).mean(), excess_over_eps=d.risk_sent.mean() - d.name))).reset_index()
print("\nP1 — abstaining policy"); print(P1.round(4).to_string(index=False))

## 10. P2 — detectability of cover admission

If the sender keeps only covers with E(X) above its q-quantile and the warden knows the rule, no single-image detector exceeds
AUC 0.5 + q/2, and thresholding E(X) attains it.

In [ ]:
# ============================================================
# 10. P2: cover-selection detectability vs q
# ============================================================
g = np.random.default_rng(77); p = g.permutation(POOL); A_, B_ = p[:HALF], p[HALF:]
rows = []
for q in CFG["admission_q_grid"]:
    thr = admit_threshold(q)
    sent_E = E_EVAL[A_][E_EVAL[A_] >= thr]; ordinary_E = E_EVAL[B_]
    q_eff = np.mean(E_EVAL[POOL] < thr)
    rows.append(dict(q=q, q_realized=q_eff, auc_bound=0.5 + q_eff / 2, auc_empirical=auc(ordinary_E, sent_E)))
P2 = pd.DataFrame(rows)
print(P2.round(4).to_string(index=False))

## 11. Mismatch margin by leave-one-detector-out (sender side only)

The sender cannot query the warden, but she can imitate the mismatch inside her own family. For each family member j she
calibrates on the other members only and measures the risk that j assigns to the images she would send (j flags with its own 5%
threshold). Repeating this on the **validation covers** (never used for calibration or testing) over random halves and internal
targets ε_int gives, for each policy and each ε, the largest ε_int whose worst held-out risk stays ≤ ε. The final sender then
calibrates on the whole family at that ε_int. The wardens play no part in this choice.

In [ ]:
# ============================================================
# 11. Leave-one-detector-out mismatch margin on validation covers
# ============================================================
vpos = POS["val"]; vpos = vpos[E_EVAL[vpos] >= thr_main]
fam = list(MODELS)
EPS_INT = np.round(np.arange(CFG["eps_int_step"], max(CFG["eps_grid"]) + 1e-9, CFG["eps_int_step"]), 4)
MARGIN = {}; LODO_ROWS = []
if len(fam) >= 2:
    LODO = {}
    for j in fam:
        rest = [n for n in fam if n != j]
        zt_j, _, _ = family_envelope(rest, POS["val"])
        tau_j = float(np.quantile(LATS[j][POS["val"], 0], 1 - CFG["cover_fa"], method="higher"))
        LODO[j] = (zt_j, LATS[j] >= tau_j)
    for r in range(CFG["n_margin_resplits"]):
        p = np.random.default_rng(30_000 + r).permutation(vpos); ca, te = p[: len(p) // 2], p[len(p) // 2:]
        for j, (zt_j, flag_j) in LODO.items():
            sels = dict(per_image=zt_j, fixed=np.broadcast_to(GRID, zt_j.shape), detectability_limited=DEFL)
            loss_j = zt_j >= 0
            for pol, S in sels.items():
                for e in EPS_INT:
                    t = crc_t(S[ca], loss_j[ca], e)
                    if t is None:
                        continue
                    k = select(np.array(S[te]), t)
                    LODO_ROWS.append(dict(resplit=r, heldout=j, policy=pol, eps_int=e,
                                          risk_heldout=flag_j[te, k].mean(), payload=GRID[k].mean()))
        if (r + 1) % 10 == 0:
            log(f"   margin resplit {r + 1}/{CFG['n_margin_resplits']}")
    LODO_DF = pd.DataFrame(LODO_ROWS)
    worst = LODO_DF.groupby(["policy", "eps_int", "heldout"]).risk_heldout.mean().groupby(["policy", "eps_int"]).max()
    for pol in SEL:
        for eps in CFG["eps_grid"]:
            w = worst.loc[pol] if pol in worst.index.get_level_values(0) else pd.Series(dtype=float)
            okk = w[(w <= eps) & (w.index <= eps + 1e-9)]   # never calibrate above eps
            MARGIN[(pol, eps)] = float(okk.index.max()) if len(okk) else None
else:
    LODO_DF = pd.DataFrame()
    for pol in SEL:
        for eps in CFG["eps_grid"]:
            MARGIN[(pol, eps)] = None
M_TAB = pd.DataFrame([dict(policy=p, eps=e, eps_int=v) for (p, e), v in MARGIN.items()])
print("Internal targets chosen by leave-one-detector-out (None = no internal target meets eps):")
print(M_TAB.to_string(index=False))

## 12. H3, M and H1 — warden risk and payload, without and with the mismatch margin

For every resplit, ε and policy, the sender calibrates on the whole family (i) at ε (**uncorrected**, H3) and (ii) at the
internal target chosen in Section 11 (**corrected**, M). The images she sends are then flagged by each warden at its own 5%
false-alarm threshold. H1 compares payloads at the corrected operating points, paired over the resplits in which both policies are feasible; intervals are 2.5–97.5 percentiles over resplits.

In [ ]:
# ============================================================
# 12. Transfer to the wardens, uncorrected and corrected; payload comparison
# ============================================================
recs = []
for r in range(CFG["n_resplits"]):
    ca, te = resplit(r, 20_000)
    ca_a, te_a = ca[E_EVAL[ca] >= thr_main], te[E_EVAL[te] >= thr_main]
    for eps in CFG["eps_grid"]:
        for pol, S in SEL.items():
            for mode, e_cal in [("uncorrected", eps), ("corrected", MARGIN[(pol, eps)])]:
                if e_cal is None:
                    continue
                t = crc_t(S[ca_a], LOSS[ca_a], e_cal)
                if t is None:
                    continue
                k = select(np.array(S[te_a]), t)
                rec = dict(resplit=r, eps=eps, policy=pol, mode=mode, eps_cal=e_cal, mean_payload=float(GRID[k].mean()),
                           risk_family=float(LOSS[te_a, k].mean()))
                for w in WARDENS:
                    rec[f"risk_{w}"] = float(W_FLAG[w][te_a, k].mean())
                rec["risk_worst_warden"] = max(rec[f"risk_{w}"] for w in WARDENS)
                recs.append(rec)
    if (r + 1) % 10 == 0:
        log(f"   transfer resplit {r + 1}/{CFG['n_resplits']}")
TR_RAW = pd.DataFrame(recs)

rows = []
for (mode, pol, eps), d in TR_RAW.groupby(["mode", "policy", "eps"]):
    row = dict(mode=mode, policy=pol, eps=eps, eps_cal=d.eps_cal.iloc[0], n=len(d),
               payload=d.mean_payload.mean(), payload_lo=d.mean_payload.quantile(.025), payload_hi=d.mean_payload.quantile(.975),
               risk_family=d.risk_family.mean())
    for w in WARDENS:
        m, se = d[f"risk_{w}"].mean(), d[f"risk_{w}"].std(ddof=1) / np.sqrt(len(d))
        row[f"risk_{w}"] = m; row[f"se_{w}"] = se
    row["worst_warden"] = d.risk_worst_warden.mean()
    row["within_eps"] = bool(row["worst_warden"] <= eps)
    row["within_eps_plus_005"] = bool(row["worst_warden"] <= eps + 0.05)
    rows.append(row)
TR = pd.DataFrame(rows).sort_values(["mode", "eps", "policy"])
print(TR.round(4).to_string(index=False))

# H1 at the corrected operating points: paired payload differences per resplit (both policies feasible)
piv = TR_RAW[TR_RAW["mode"] == "corrected"].pivot_table(index=["resplit", "eps"], columns="policy", values="mean_payload")
wr = TR_RAW[TR_RAW["mode"] == "corrected"].pivot_table(index=["resplit", "eps"], columns="policy", values="risk_worst_warden")
PAIRS = [("detectability_limited", "fixed"), ("per_image", "fixed"), ("per_image", "detectability_limited")]
rows = []
for eps in CFG["eps_grid"]:
    for a, b in PAIRS:
        if eps not in piv.index.get_level_values("eps") or a not in piv or b not in piv:
            rows.append(dict(eps=eps, policy=a, baseline=b, n_paired=0)); continue
        d = piv.xs(eps, level="eps")[[a, b]].dropna(); r = wr.xs(eps, level="eps")[[a, b]].loc[d.index]
        if len(d) == 0:
            rows.append(dict(eps=eps, policy=a, baseline=b, n_paired=0)); continue
        diff = d[a] - d[b]
        rows.append(dict(eps=eps, policy=a, baseline=b, n_paired=len(d), policy_bpp=d[a].mean(), baseline_bpp=d[b].mean(),
                         gain_bpp=diff.mean(), gain_lo=diff.quantile(.025), gain_hi=diff.quantile(.975),
                         ratio=d[a].mean() / max(d[b].mean(), 1e-9),
                         worst_warden_policy=r[a].mean(), worst_warden_baseline=r[b].mean()))
H1 = pd.DataFrame(rows)
print("\nH1 — paired payload gains at the corrected operating points (resplits where both policies are feasible)")
print(H1.round(4).to_string(index=False))

In [ ]:
# ============================================================
# 13. Figures, LaTeX tables, JSON; copy small results
# ============================================================
FIG = WORK_DIR / "figures"; TAB = WORK_DIR / "tables"
FIG.mkdir(exist_ok=True); TAB.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(4.8, 3.3))
for n, d in NET_AUC.groupby("model"):
    ax.plot(d.payload, d.auc, marker="o", ms=2.5, ls="-" if d.role.iloc[0] == "family" else "--", label=n)
ax.set_xlabel("Payload (bpp)"); ax.set_ylabel("AUC, cover vs stego (test)"); ax.grid(alpha=.3); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "network_auc_vs_payload.pdf"); fig.savefig(FIG / "network_auc_vs_payload.png", dpi=200); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(4.8, 3.3))
for pol, d in G_RAW[G_RAW.feasible == True].groupby("policy"):
    m = d.groupby("eps").risk_all.mean(); s = d.groupby("eps").risk_all.std()
    ax.errorbar(m.index, m.values, yerr=s.values, marker="o", capsize=3, label=pol)
e = np.array(CFG["eps_grid"]); ax.plot(e, e, "k--", lw=1, label="target ε")
ax.set_xlabel("Target risk ε"); ax.set_ylabel("Family risk (mean ± sd)"); ax.grid(alpha=.3); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "G_risk_control.pdf"); fig.savefig(FIG / "G_risk_control.png", dpi=200); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(4.8, 3.3))
ax.plot(P2.q_realized, P2.auc_bound, "k--", label="bound 0.5 + q/2")
ax.plot(P2.q_realized, P2.auc_empirical, "o-", label="residual energy, empirical")
ax.set_xlabel("Discarded fraction q"); ax.set_ylabel("Single-image AUC"); ax.grid(alpha=.3); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "P2_selection_bias.pdf"); fig.savefig(FIG / "P2_selection_bias.png", dpi=200); plt.show(); plt.close(fig)

modes = [m for m in ["uncorrected", "corrected"] if m in set(TR["mode"])]
fig, axes = plt.subplots(1, len(modes), figsize=(4.4 * len(modes), 3.3), sharey=True); axes = np.atleast_1d(axes)
for ax, mode in zip(axes, modes):
    for pol, d in TR[TR["mode"] == mode].groupby("policy"):
        ax.plot(d.eps, d.worst_warden, marker="o", label=pol)
    ax.plot(e, e, "k--", lw=1, label="ε"); ax.set_title(mode, fontsize=9); ax.set_xlabel("Target risk ε"); ax.grid(alpha=.3)
axes[0].set_ylabel("Worst warden risk on sent images"); axes[-1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "transfer_worst_warden.pdf"); fig.savefig(FIG / "transfer_worst_warden.png", dpi=200); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(4.8, 3.3))
for pol, d in TR[TR["mode"] == "corrected"].groupby("policy"):
    ax.errorbar(d.worst_warden, d.payload, yerr=[d.payload - d.payload_lo, d.payload_hi - d.payload], marker="o", capsize=3, label=pol)
ax.set_xlabel("Worst warden risk"); ax.set_ylabel("Mean payload (bpp)"); ax.grid(alpha=.3); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "H1_payload_vs_warden_risk.pdf"); fig.savefig(FIG / "H1_payload_vs_warden_risk.png", dpi=200); plt.show(); plt.close(fig)

for name, df in dict(G_summary=G_SUM, P1_abstention=P1, P2_selection_bias=P2, M_margin=M_TAB, transfer=TR, H1_corrected=H1).items():
    df.to_latex(TAB / f"{name}.tex", index=False, float_format="%.4f")
if len(LODO_DF):
    LODO_DF.groupby(["policy", "eps_int", "heldout"]).agg(risk=("risk_heldout", "mean"), payload=("payload", "mean")).reset_index() \
        .to_csv(TAB / "M_lodo_curves.csv", index=False)

RESULTS = dict(config=CFG, family=list(MODELS), wardens=list(WARDENS), tau=TAU,
               standardization={k: list(v) for k, v in STD.items()}, warden_thresholds=W_TAU,
               network_auc=NET_AUC.to_dict("records"), G=G_SUM.to_dict("records"), P1=P1.to_dict("records"),
               P2=P2.to_dict("records"), margin=M_TAB.to_dict("records"), transfer=TR.to_dict("records"),
               H1=H1.to_dict("records"), runtime_min=(time.time() - T0) / 60)
(WORK_DIR / "full_results.json").write_text(json.dumps(RESULTS, indent=2, default=str))

try:
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    for src in [WORK_DIR / "full_results.json", *FIG.glob("*"), *TAB.glob("*"), *WORK_DIR.glob("train_history_*.csv")]:
        dst = RESULT_DIR / src.relative_to(WORK_DIR); dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
    log(f"Small results copied to {RESULT_DIR}")
except Exception as ex:
    log(f"Could not copy to {RESULT_DIR} ({ex}); results remain in {WORK_DIR}")
log("Done.")